# STRAP on MNIST-C with a LeNet-type convolutional autoencoder (calibrated prediction-interval test)

**Protocol** (Ruff et al., 2021): autoencoders are trained on clean MNIST training digits; each of the 15 MNIST-C corruptions is a separate evaluation with the 10,000 clean MNIST test digits as normal samples and their 10,000 corrupted versions as anomalies (50% anomalies).

**Data:** MNIST-C (Mu & Gilmer, 2019) from its official Zenodo record, as referenced by the `google-research/mnist-c` GitHub repository. The archive's `identity` folder holds the clean MNIST training and test digits. The archive is cached in Drive after the first download.

**STRAP setup:**
- **Base autoencoder, following Ruff et al. (2021):** "a LeNet-type encoder that has two convolutional layers with max-pooling followed by two fully connected layers that map to an encoding of 64 dimensions, and construct the decoder symmetrically." Channel and hidden sizes are not given in the paper; we use conv 5×5 with 32 and 64 channels, a 256-unit hidden layer and a 64-dimensional code.
- **Training, following Ruff et al.:** "we train the AE such that the reconstruction error of a small training hold-out set is minimized", i.e. early stopping on a validation split (5% of the fit set, patience 10, at most 200 epochs, best weights restored). Batch 256, Adam (lr 1e-3), loss MSE + λ·mean|z|.
- λ ∈ {0, 1e-4, 1e-3, 1e-2, 1e-1}; seeds 0, 1, 2.
- **Calibration set:** a random 20% of the training digits (12,000) is held out per seed and never seen by the autoencoders, which train on the remaining 48,000.
- **Decision test:** per λ, Box-Cox fitted on the calibration errors and the one-sided prediction-interval statistic t = (y − ȳ)/(s·√(1+1/n)) against Student-t(n−1). Rules: λ=0 only, union, ≥2, majority (≥3), all 5, Bonferroni union. Ranking score: maximum t over the path.

Each autoencoder is trained once per (seed, λ) and evaluated on all 15 corruptions, so there are 15 trainings in total. Runs are resumable: a run is skipped if its output file exists.

**Saved per run:** reconstruction errors of the calibration digits, a training subsample, the clean test digits and every corruption. The analysis cell computes everything from these files.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, time, zipfile, subprocess, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats
from scipy.special import boxcox as boxcox_transform
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

In [ ]:
# ── Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/STRAP_MNISTC'
RUNS_DIR   = os.path.join(DRIVE_ROOT, 'runs_conv_cal')
ZIP_CACHE  = os.path.join(DRIVE_ROOT, 'mnist_c.zip')
DATA_DIR   = '/content/mnist_c'
os.makedirs(RUNS_DIR, exist_ok=True)
print('Outputs:', DRIVE_ROOT)

In [ ]:
# ── Data: MNIST-C from Zenodo (cached in Drive) ────────────────────────
ZENODO = 'https://zenodo.org/records/3239543/files/mnist_c.zip?download=1'
CORRUPTIONS = ['shot_noise', 'impulse_noise', 'glass_blur', 'motion_blur', 'shear', 'scale', 'rotate',
               'brightness', 'translate', 'stripe', 'fog', 'spatter', 'dotted_line', 'zigzag', 'canny_edges']
if not os.path.isdir(os.path.join(DATA_DIR, 'identity')):
    if not os.path.exists(ZIP_CACHE):
        print('downloading MNIST-C from Zenodo ...')
        subprocess.run(['wget', '-q', '-O', ZIP_CACHE, ZENODO], check=True)
    with zipfile.ZipFile(ZIP_CACHE) as zf: zf.extractall('/content')
    assert os.path.isdir(os.path.join(DATA_DIR, 'identity')), 'unexpected archive layout'

def load(folder, part):
    x = np.load(os.path.join(DATA_DIR, folder, f'{part}_images.npy'))
    return (x.reshape(len(x), -1).astype(np.float32) / 255.0)

X_TRAIN = load('identity', 'train')                   # 60,000 clean training digits
X_NORMAL = load('identity', 'test')                   # 10,000 clean test digits (normal)
X_CORR = {c: load(c, 'test') for c in CORRUPTIONS}    # 10,000 corrupted test digits each (anomalies)
assert X_TRAIN.shape == (60000, 784) and X_NORMAL.shape == (10000, 784)
assert all(v.shape == (10000, 784) for v in X_CORR.values())
print('train', X_TRAIN.shape, '| normal test', X_NORMAL.shape, '| corruptions', len(X_CORR))

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
LAMBDAS  = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
SEEDS    = [0, 1, 2]
MAX_EPOCHS, PATIENCE, VAL_FRAC = 200, 10, 0.05   # early stopping on a small hold-out (Ruff et al.)
BATCH, LR = 256, 1e-3
LATENT = 64
CAL_FRAC = 0.20
ALPHAS   = [0.05, 0.10]
def lam_tag(l): return 'lam0' if l == 0 else f'lam1e{abs(int(round(np.log10(l))))}'
def run_file(seed, lam): return os.path.join(RUNS_DIR, f'mnistc_conv_seed{seed}_{lam_tag(lam)}.npz')
print(len(SEEDS) * len(LAMBDAS), 'trainings')

In [ ]:
# ── Model, training, errors ────────────────────────────────────────────
class AE(nn.Module):
    # LeNet-type convolutional autoencoder (Ruff et al., 2021): two conv layers with max-pooling,
    # two fully connected layers to a 64-d code; decoder symmetric.
    def __init__(self, latent=LATENT):
        super().__init__()
        act = lambda: nn.LeakyReLU(0.1)
        self.enc_conv = nn.Sequential(
            nn.Conv2d(1, 32, 5, padding=2), nn.BatchNorm2d(32), act(), nn.MaxPool2d(2),     # 28 -> 14
            nn.Conv2d(32, 64, 5, padding=2), nn.BatchNorm2d(64), act(), nn.MaxPool2d(2))    # 14 -> 7
        self.enc_fc = nn.Sequential(nn.Flatten(), nn.Linear(64 * 7 * 7, 256), nn.BatchNorm1d(256), act(),
                                    nn.Linear(256, latent))
        self.dec_fc = nn.Sequential(nn.Linear(latent, 256), nn.BatchNorm1d(256), act(),
                                    nn.Linear(256, 64 * 7 * 7), nn.BatchNorm1d(64 * 7 * 7), act())
        self.dec_conv = nn.Sequential(
            nn.Upsample(scale_factor=2), nn.Conv2d(64, 32, 5, padding=2), nn.BatchNorm2d(32), act(),   # 7 -> 14
            nn.Upsample(scale_factor=2), nn.Conv2d(32, 1, 5, padding=2), nn.Sigmoid())                # 14 -> 28
    def forward(self, x):
        z = self.enc_fc(self.enc_conv(x.view(-1, 1, 28, 28)))
        xh = self.dec_conv(self.dec_fc(z).view(-1, 64, 7, 7))
        return xh.view(len(x), -1), z

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def mean_mse(net, X, batch=8192):
    net.eval(); tot = 0.0
    for i in range(0, len(X), batch):
        xb = torch.from_numpy(X[i:i + batch]).to(DEVICE); tot += ((xb - net(xb)[0]) ** 2).mean(1).sum().item()
    return tot / len(X)

def train(X, lam, seed):
    # early stopping on a small hold-out of the fit set (Ruff et al.: minimize hold-out reconstruction error)
    perm = np.random.default_rng(20_000 + seed).permutation(len(X)); n_val = int(round(VAL_FRAC * len(X)))
    X_val, X_tr = X[perm[:n_val]], X[perm[n_val:]]
    net = AE().to(DEVICE); opt = torch.optim.Adam(net.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(torch.from_numpy(X_tr)), batch_size=BATCH, shuffle=True, drop_last=True)
    best, best_state, wait = np.inf, None, 0
    for epoch in range(MAX_EPOCHS):
        net.train()
        for (xb,) in loader:
            xb = xb.to(DEVICE); xh, z = net(xb)
            loss = nn.functional.mse_loss(xh, xb) + lam * z.abs().mean()
            opt.zero_grad(); loss.backward(); opt.step()
        val = mean_mse(net, X_val)
        if val < best - 1e-7: best, wait = val, 0; best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break
    net.load_state_dict(best_state)
    return net, epoch + 1

@torch.no_grad()
def errors(net, X, batch=8192):
    net.eval(); out = []
    for i in range(0, len(X), batch):
        xb = torch.from_numpy(X[i:i + batch]).to(DEVICE); out.append(((xb - net(xb)[0]) ** 2).mean(1).cpu().numpy())
    return np.concatenate(out)
print('ready')

In [ ]:
# ── Main loop (resumable) ──────────────────────────────────────────────
for seed in SEEDS:
    perm = np.random.default_rng(10_000 + seed).permutation(len(X_TRAIN))
    n_cal = int(round(CAL_FRAC * len(X_TRAIN)))
    X_fit, X_cal = X_TRAIN[perm[n_cal:]], X_TRAIN[perm[:n_cal]]
    for lam in LAMBDAS:
        if os.path.exists(run_file(seed, lam)): continue
        t0 = time.time(); set_seed(seed)
        net, n_epochs = train(X_fit, lam, seed)
        sub = np.random.default_rng(seed).choice(len(X_fit), 5000, replace=False)
        np.savez_compressed(run_file(seed, lam),
            cal_err=errors(net, X_cal).astype(np.float32),
            train_err_sample=errors(net, X_fit[sub]).astype(np.float32),
            normal_err=errors(net, X_NORMAL).astype(np.float32),
            corr_err=np.stack([errors(net, X_CORR[c]) for c in CORRUPTIONS]).astype(np.float32),
            corruptions=np.array(CORRUPTIONS), n_epochs=n_epochs)
        print(f'seed {seed} {lam_tag(lam)} done: {n_epochs} epochs ({time.time() - t0:.0f}s)')
        del net; torch.cuda.empty_cache()
print('Done.')

In [ ]:
# ── Analysis ───────────────────────────────────────────────────────────
def pi_stat(ref, e):
    rb, lmb = stats.boxcox(np.maximum(ref, 1e-12)); eb = boxcox_transform(np.maximum(e, 1e-12), lmb)
    n = len(rb); return (eb - rb.mean()) / (rb.std(ddof=1) * np.sqrt(1 + 1 / n)), n - 1

def prf(y, det):
    tp = int((det & (y == 1)).sum()); fp = int((det & (y == 0)).sum())
    p = tp / max(1, int(det.sum())); r = tp / max(1, int(y.sum()))
    return {'f1': 0.0 if p + r == 0 else 2 * p * r / (p + r), 'precision': p, 'recall': r, 'fpr': fp / max(1, int((y == 0).sum()))}

rows, norm_rows = [], []
rng = np.random.default_rng(0)
K = len(LAMBDAS)
for seed in SEEDS:
    files = [run_file(seed, l) for l in LAMBDAS]
    if not all(os.path.exists(f) for f in files): continue
    R = [np.load(f) for f in files]; corrs = list(R[0]['corruptions'])
    cal = [r['cal_err'].astype(np.float64) for r in R]
    for k, l in enumerate(LAMBDAS):                      # normality of calibration errors, raw and Box-Cox
        x = rng.choice(cal[k], 5000, replace=False); xb, _ = stats.boxcox(np.maximum(x, 1e-12))
        norm_rows.append({'seed': seed, 'lambda': l, 'sw_p_raw': stats.shapiro(x).pvalue, 'ad_raw': stats.anderson(x).statistic,
                          'skew_raw': stats.skew(x), 'sw_p_boxcox': stats.shapiro(xb).pvalue, 'ad_boxcox': stats.anderson(xb).statistic})
    for j, c in enumerate(corrs):
        y = np.r_[np.zeros(10000, int), np.ones(10000, int)]
        E = [np.r_[r['normal_err'], r['corr_err'][j]].astype(np.float64) for r in R]
        TT, DF = zip(*[pi_stat(cal[k], E[k]) for k in range(K)])
        TT = np.stack(TT); P = np.stack([stats.t.sf(TT[k], DF[k]) for k in range(K)])
        m = {'corruption': c, 'seed': seed, 'auroc': roc_auc_score(y, TT.max(0)), 'ap': average_precision_score(y, TT.max(0)),
             'auroc_raw_lam0': roc_auc_score(y, E[0]), 'ap_raw_lam0': average_precision_score(y, E[0])}
        for a in ALPHAS:
            D = P < a; v = D.sum(0)
            for r, det in {'lam0': D[0], 'union': v >= 1, 'ge2': v >= 2, 'majority': v >= 3, 'all': v == K,
                           'bonferroni': (P < a / K).any(0)}.items():
                for k2, val in prf(y, det).items(): m[f'{k2}_{r}_a{a}'] = val
            for k in range(K): m[f'fpr_{lam_tag(LAMBDAS[k])}_a{a}'] = float(D[k][y == 0].mean())
        rows.append(m)
per_seed = pd.DataFrame(rows); norm = pd.DataFrame(norm_rows)
summary = per_seed.drop(columns='seed').groupby('corruption').agg(['mean', 'std'])
summary.columns = [f'{a}_{b}' for a, b in summary.columns]
per_seed.to_csv(os.path.join(DRIVE_ROOT, 'mnistc_per_seed.csv'), index=False)
summary.reset_index().to_csv(os.path.join(DRIVE_ROOT, 'mnistc_summary.csv'), index=False)
norm.to_csv(os.path.join(DRIVE_ROOT, 'mnistc_normality.csv'), index=False)
show = ['auroc', 'ap', 'auroc_raw_lam0', 'f1_majority_a0.05', 'f1_union_a0.05', 'fpr_lam0_a0.05', 'fpr_majority_a0.05', 'fpr_union_a0.05']
t = 100 * summary[[f'{c}_mean' for c in show]]; t.loc['MEAN'] = t.mean()
print(t.round(1).to_string())
print(); print('Normality of calibration errors (pass rate at 5%):',
      f"raw SW {(norm.sw_p_raw > .05).mean():.2f}, Box-Cox SW {(norm.sw_p_boxcox > .05).mean():.2f}")